In [1]:
from datetime import datetime
from mlflow.entities import ViewType
from mlflow.tracking import MlflowClient
import mlflow

MLFLOW_TRACKING_URI = "../../mlruns/"

client = MlflowClient(tracking_uri=MLFLOW_TRACKING_URI)

In [2]:
client.search_experiments()

[<Experiment: artifact_location='/home/irb/Documents/mlops-zoomcamp/02-training/model_usage/../experiment_tracking/mlruns/341114581205668630', creation_time=1778110291510, experiment_id='341114581205668630', last_update_time=1778110291510, lifecycle_stage='active', name='my-cool-experiment', tags={}>,
 <Experiment: artifact_location='file:///home/irb/Documents/mlops-zoomcamp/mlruns/284734417574169759', creation_time=1777831253476, experiment_id='284734417574169759', last_update_time=1777831253476, lifecycle_stage='active', name='nyc-taxi', tags={}>,
 <Experiment: artifact_location='file:///home/irb/Documents/mlops-zoomcamp/mlruns/0', creation_time=1777831253473, experiment_id='0', last_update_time=1777831253473, lifecycle_stage='active', name='Default', tags={}>]

In [3]:
# Already created once!
# client.create_experiment(name="my-cool-experiment")

In [4]:
runs = client.search_runs(
    experiment_ids="284734417574169759",
    filter_string="metrics.rmse < 6.5",
    run_view_type=ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=["metrics.rmse ASC"]
)

In [5]:
for run in runs:
    print(f"run id: {run.info.run_id}, rmse: {run.data.metrics['rmse']:.4f}")

run id: eed46ccfa3a444739b55e648a0418719, rmse: 6.3940
run id: 40b1eac36cb442049b98e5b1432b9dbb, rmse: 6.4045
run id: 6c54b73a766740fba071520c01b9e087, rmse: 6.4074
run id: f7e94c05c806415fa23b8728d10488d4, rmse: 6.4081
run id: 9cbc639956cd4954b6f56da6c2cce98a, rmse: 6.4086
run id: 40a54bf430c343878ba69b7b6d2069fd, rmse: 6.4185
run id: 4ec4a0e027804a50ae382860f848ce65, rmse: 6.4196
run id: 5229d7e6609648fd8ae0cf81928558d3, rmse: 6.4237
run id: d80143d3fec841a9a6657d89b791b7fc, rmse: 6.4240
run id: 497a2729668b47efb243f54395b04876, rmse: 6.4303


In [6]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)

In [7]:
model_name = "nyc-taxi-regressor"
model_alias = "champion"
run_id = "eed46ccfa3a444739b55e648a0418719"
model_uri = f"runs:/{run_id}/model"
mlflow.register_model(model_uri=model_uri, name=model_name)

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '6' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1784567667734, current_stage='None', description=None, last_updated_timestamp=1784567667734, name='nyc-taxi-regressor', run_id='eed46ccfa3a444739b55e648a0418719', run_link=None, source='file:///home/irb/Documents/mlops-zoomcamp/mlruns/284734417574169759/eed46ccfa3a444739b55e648a0418719/artifacts/model', status='READY', status_message=None, tags={}, user_id=None, version=6>

In [8]:
client.get_model_version_by_alias(name=model_name, alias=model_alias)

<ModelVersion: aliases=['champion'], creation_timestamp=1778109769473, current_stage='Production', description='The model version 1 was transitioned to production on 2026-07-20.', last_updated_timestamp=1784567624707, name='nyc-taxi-regressor', run_id='eed46ccfa3a444739b55e648a0418719', run_link='', source='file:///home/irb/Documents/mlops-zoomcamp/mlruns/284734417574169759/eed46ccfa3a444739b55e648a0418719/artifacts/models_mlflow', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=1>

In [9]:
latest_versions = client.get_latest_versions(name=model_name)

for version in latest_versions:
    print(f"version: {version.version}, stage: {version.current_stage}")

version: 1, stage: Production
version: 6, stage: None


/tmp/ipykernel_35922/232761966.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_versions = client.get_latest_versions(name=model_name)


In [10]:
# MLflow stages is deprecated since 2.9.0
model_version = 1
new_stage = "production"

client.transition_model_version_stage(
    name=model_name,
    version=model_version,
    stage=new_stage,
    archive_existing_versions=False
)

/tmp/ipykernel_35922/1719677172.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=['champion'], creation_timestamp=1778109769473, current_stage='Production', description='The model version 1 was transitioned to production on 2026-07-20.', last_updated_timestamp=1784567667743, name='nyc-taxi-regressor', run_id='eed46ccfa3a444739b55e648a0418719', run_link='', source='file:///home/irb/Documents/mlops-zoomcamp/mlruns/284734417574169759/eed46ccfa3a444739b55e648a0418719/artifacts/models_mlflow', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=1>

In [11]:
date = datetime.today().date()
client.update_model_version(
    name=model_name,
    version=model_version,
    description=f"The model version {model_version} was transitioned to {new_stage} on {date}."
)

<ModelVersion: aliases=['champion'], creation_timestamp=1778109769473, current_stage='Production', description='The model version 1 was transitioned to production on 2026-07-20.', last_updated_timestamp=1784567667747, name='nyc-taxi-regressor', run_id='eed46ccfa3a444739b55e648a0418719', run_link='', source='file:///home/irb/Documents/mlops-zoomcamp/mlruns/284734417574169759/eed46ccfa3a444739b55e648a0418719/artifacts/models_mlflow', status='READY', status_message=None, tags={'model': 'xgboost'}, user_id=None, version=1>

In [12]:
import pandas as pd

from sklearn.metrics import root_mean_squared_error

In [13]:
def read_dataframe(filename):
    df = pd.read_parquet(filename)

    df['duration'] = df.lpep_dropoff_datetime - df.lpep_pickup_datetime
    df.duration = df.duration.apply(lambda td: td.total_seconds()/60)
    
    df = df[(df.duration >= 1) & (df.duration <= 60)]
    
    categorical = ['PULocationID', 'DOLocationID']
    df[categorical] = df[categorical].astype(str)

    return df

def preprocess(df, dv):
    df['PU_DO'] = df['PULocationID'] + '_' + df['DOLocationID']
    categorical = ['PU_DO']
    numerical = ['trip_distance']
    train_dicts = df[categorical + numerical].to_dict(orient='records')

    return dv.transform(train_dicts)

def test_model(name, stage, x_test, y_test):
    model = mlflow.pyfunc.load_model(f"models:/{name}/{stage}")
    y_pred = model.predict(x_test)
    return {"rmse": root_mean_squared_error(y_test, y_pred)}

In [14]:
df = read_dataframe("../../data/green_tripdata_2021-03.parquet")

In [15]:
client.download_artifacts(run_id=run_id, path='preprocessor', dst_path='.')

'/home/irb/Documents/mlops-zoomcamp/02-training/model_usage/preprocessor'

In [16]:
import pickle

with open("preprocessor/preprocessor.py", "rb") as f_in:
    dv = pickle.load(f_in)

In [17]:
x_test = preprocess(df, dv)

In [18]:
target = "duration"
y_test = df[target].values

In [19]:
%time test_model(name=model_name, stage="Production", x_test=x_test, y_test=y_test)

/home/irb/Documents/mlops-zoomcamp/.venv/lib/python3.12/site-packages/mlflow/xgboost/__init__.py:302: UserWarning: [14:14:28] WARNING: /__w/xgboost/xgboost/src/c_api/c_api.cc:1509: Unknown file format: `xgb`. Using UBJSON (`ubj`) as a guess.
  model.load_model(xgb_model_path)


CPU times: user 8.28 s, sys: 16.3 ms, total: 8.3 s
Wall time: 432 ms


{'rmse': 6.316282862727709}